In [ ]:
# =========================================================
# 50-State TSP Visual App  (my student version)
# Nearest Neighbor vs Genetic Algorithm (OX / PMX)
# Distances are approximate real-world miles using lat/lon
# =========================================================

import tkinter as tk
from tkinter import ttk, messagebox

import csv
import math
import random
import time
from typing import List, Tuple

import numpy as np
import matplotlib
matplotlib.use("TkAgg")   # show matplotlib figures inside the Tk window
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

import warnings
warnings.filterwarnings("ignore")   # ignore harmless matplotlib warnings


# =========================================================
# 1. Data loading + distance in miles
# =========================================================

def load_states(path: str) -> Tuple[List[str], List[Tuple[float, float]]]:
    """
    Read the CSV file and return:
      - list of state names
      - list of (lat, lon) coordinates
    """
    names: List[str] = []
    coords: List[Tuple[float, float]] = []

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            names.append(row["state"])
            coords.append((float(row["lat"]), float(row["lon"])))
    return names, coords


def euclidean_miles(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
    """
    Approximate real-world distance in miles from (lat, lon) pairs.
    Assumptions:
      - 1 degree latitude ≈ 69 miles
      - 1 degree longitude ≈ 69 * cos(latitude) miles
    """
    lat1, lon1 = p1
    lat2, lon2 = p2

    avg_lat = (lat1 + lat2) / 2.0
    avg_lat_rad = math.radians(avg_lat)

    miles_per_lat = 69.0
    miles_per_lon = 69.0 * math.cos(avg_lat_rad)

    dy = (lat2 - lat1) * miles_per_lat
    dx = (lon2 - lon1) * miles_per_lon

    return math.hypot(dx, dy)


def build_distance_matrix(coords: List[Tuple[float, float]]) -> np.ndarray:
    """Build an N×N matrix of pairwise distances in miles."""
    n = len(coords)
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            D[i, j] = euclidean_miles(coords[i], coords[j])
    return D


def path_length(route: List[int], D: np.ndarray, round_trip: bool) -> float:
    """Compute total distance in miles for a route (optionally round trip)."""
    total = 0.0
    for i in range(len(route) - 1):
        total += D[route[i], route[i + 1]]
    if round_trip and len(route) > 1:
        total += D[route[-1], route[0]]
    return total


def nn_convergence(route: List[int],
                   D: np.ndarray,
                   round_trip: bool) -> List[float]:
    """
    Track cumulative distance for the NN route as we add edges,
    to get a "convergence" style curve.
    """
    cum = []
    running = 0.0

    for i in range(len(route) - 1):
        running += D[route[i], route[i + 1]]
        cum.append(running)

    if round_trip and len(route) > 1:
        running += D[route[-1], route[0]]
        cum.append(running)

    return cum


# =========================================================
# 2. Nearest Neighbor heuristic
# =========================================================

def nearest_neighbor(D: np.ndarray,
                     start: int,
                     end: int,
                     round_trip: bool) -> List[int]:
    """
    Simple greedy nearest neighbor.
    If round_trip is False, I keep the chosen end city aside and visit it last.
    """
    n = D.shape[0]
    remaining = set(range(n))
    remaining.discard(start)

    route = [start]
    current = start

    keep_end_for_last = False
    if not round_trip and end in remaining:
        remaining.discard(end)
        keep_end_for_last = True

    while remaining:
        next_city = min(remaining, key=lambda j: D[current, j])
        route.append(next_city)
        remaining.discard(next_city)
        current = next_city

    if keep_end_for_last:
        route.append(end)

    return route


# =========================================================
# 3. Genetic Algorithm helpers (with faster PMX)
# =========================================================

def init_population(pop_size: int,
                    n_cities: int,
                    start: int,
                    end: int,
                    round_trip: bool) -> List[List[int]]:
    """
    Create initial GA population.
    Start city is fixed at index 0, end city fixed at last index for point-to-point.
    """
    all_nodes = list(range(n_cities))
    population: List[List[int]] = []

    for _ in range(pop_size):
        if round_trip:
            middle = [c for c in all_nodes if c != start]
            random.shuffle(middle)
            chrom = [start] + middle
        else:
            middle = [c for c in all_nodes if c not in (start, end)]
            random.shuffle(middle)
            chrom = [start] + middle + [end]
        population.append(chrom)

    return population


def order_crossover(p1: List[int],
                    p2: List[int],
                    start: int,
                    end: int,
                    round_trip: bool) -> List[int]:
    """
    Ordered Crossover (OX) with fixed endpoints.
    Copy a random middle slice from p1, then fill the rest from p2 in order.
    """
    n = len(p1)
    child = [None] * n

    left_bound = 1
    right_bound = n - 1 if round_trip else n - 2

    i, j = sorted(random.sample(range(left_bound, right_bound + 1), 2))

    # copy slice from p1
    child[i:j + 1] = p1[i:j + 1]

    # fill remaining positions from p2, skipping duplicates
    used = set(c for c in child if c is not None)
    idx2 = 0
    for k in range(left_bound, right_bound + 1):
        if child[k] is None:
            while p2[idx2] in used:
                idx2 += 1
            child[k] = p2[idx2]
            used.add(p2[idx2])

    # enforce endpoints
    child[0] = start
    if not round_trip:
        child[-1] = end

    # safety: if anything is still None, fill from p1
    for k in range(n):
        if child[k] is None:
            for city in p1:
                if city not in used:
                    child[k] = city
                    used.add(city)
                    break

    return child


def pmx_crossover(p1: List[int],
                  p2: List[int],
                  start: int,
                  end: int,
                  round_trip: bool) -> List[int]:
    """
    Partially Mapped Crossover (PMX) for permutations, with fixed endpoints.
    This is a faster, cleaned up implementation.
    """
    n = len(p1)
    child = [None] * n

    left_bound = 1
    right_bound = n - 1 if round_trip else n - 2

    a, b = sorted(random.sample(range(left_bound, right_bound + 1), 2))

    # 1) Copy slice from p1
    child[a:b + 1] = p1[a:b + 1]

    # 2) Build mapping from p2 -> p1 for the slice
    mapping = {p2[i]: p1[i] for i in range(a, b + 1)}

    # track which cities are already used
    used = set(child[a:b + 1])

    # 3) Fill positions outside the slice
    for i in range(n):
        if a <= i <= b:
            continue

        gene = p2[i]

        # resolve mapping conflicts
        visited = set()
        while gene in mapping and gene not in visited:
            visited.add(gene)
            gene = mapping[gene]

        # if not already used, place it here
        if gene not in used:
            child[i] = gene
            used.add(gene)
        # else, we leave it as None for now and fill later

    # 4) Fill remaining None positions with any missing cities
    missing = [city for city in p1 if city not in used]
    idx_missing = 0
    for i in range(n):
        if child[i] is None:
            child[i] = missing[idx_missing]
            idx_missing += 1

    # 5) Fix endpoints
    child[0] = start
    if not round_trip:
        child[-1] = end

    return child


def mutate_swap(chrom: List[int],
                rate: float,
                start: int,
                end: int,
                round_trip: bool) -> None:
    """Swap mutation: randomly swap two non-endpoint positions."""
    if random.random() < rate:
        if round_trip:
            valid = list(range(1, len(chrom)))
        else:
            valid = list(range(1, len(chrom) - 1))

        if len(valid) >= 2:
            i, j = random.sample(valid, 2)
            chrom[i], chrom[j] = chrom[j], chrom[i]


def tournament_selection(pop: List[List[int]],
                         fitness: List[float],
                         k: int = 3) -> List[int]:
    """Tournament selection: pick k random individuals, return the best."""
    indices = random.sample(range(len(pop)), k)
    best_idx = max(indices, key=lambda idx: fitness[idx])
    return pop[best_idx][:]


def run_ga(D: np.ndarray,
           start: int,
           end: int,
           round_trip: bool,
           pop_size: int,
           generations: int,
           mutation_rate: float,
           crossover_type: str,
           seed: int = None):
    """
    Core GA loop.
    Returns:
      - best route
      - best distance in miles
      - history of best distance vs generation
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    n_cities = D.shape[0]
    population = init_population(pop_size, n_cities, start, end, round_trip)

    def evaluate(pop):
        lengths = [path_length(ch, D, round_trip) for ch in pop]
        fit_vals = [1.0 / (1.0 + L) for L in lengths]
        return lengths, fit_vals

    best_route = None
    best_len = float("inf")
    history: List[float] = []

    for _ in range(generations):
        lengths, fitness_vals = evaluate(population)

        # best individual of this generation
        gen_best_idx = min(range(len(population)), key=lambda idx: lengths[idx])
        gen_best_len = lengths[gen_best_idx]

        if gen_best_len < best_len:
            best_len = gen_best_len
            best_route = population[gen_best_idx][:]

        history.append(best_len)

        # elitism: carry best individual
        elite = population[gen_best_idx][:]
        new_pop: List[List[int]] = [elite]

        # fill rest of population
        while len(new_pop) < pop_size:
            p1 = tournament_selection(population, fitness_vals)
            p2 = tournament_selection(population, fitness_vals)

            if crossover_type == "PMX":
                child = pmx_crossover(p1, p2, start, end, round_trip)
            else:
                child = order_crossover(p1, p2, start, end, round_trip)

            mutate_swap(child, mutation_rate, start, end, round_trip)
            new_pop.append(child)

        population = new_pop

    return best_route, best_len, history


# =========================================================
# 4. Tkinter GUI
# =========================================================

class TSPGui:
    """
    Simple GUI that ties everything together:
      - data
      - NN heuristic
      - GA (OX / PMX)
      - plotting and summary
    """
    def __init__(self, root: tk.Tk):
        self.root = root
        root.title("50-State TSP – NN vs GA (Student Version)")

        # load data once and build distance matrix
        self.states, self.coords = load_states("us_states_50.csv")
        self.D = build_distance_matrix(self.coords)

        # NN fields
        self.nn_route = None
        self.nn_miles = None
        self.nn_time = None
        self.nn_curve = None
        self.nn_fit = None

        # GA fields
        self.ga_route = None
        self.ga_miles = None
        self.ga_hist = None
        self.ga_fit = None
        self.ga_times: List[float] = []
        self.ga_dists: List[float] = []

        # general settings
        self.start_idx = 0
        self.end_idx = 0
        self.round_trip = True
        self.cost_per_mile = 1.0
        self.speed = 60.0

        # layout
        main = ttk.Frame(root, padding=10)
        main.grid(row=0, column=0, sticky="nsew")

        self.left = ttk.Frame(main, padding=(0, 0, 10, 0))
        self.left.grid(row=0, column=0, sticky="ns")

        self.plot_area = ttk.Frame(main)
        self.plot_area.grid(row=0, column=1, sticky="nsew")

        main.grid_columnconfigure(1, weight=1)
        main.grid_rowconfigure(0, weight=1)

        self.canvas = None
        self._build_controls()
        self._reset_plot_area()

    # ---------------- controls ----------------

    def _build_controls(self):
        r = 0

        ttk.Label(self.left, text="Start state:").grid(row=r, column=0, sticky="w")
        self.start_var = tk.StringVar(value=self.states[0])
        ttk.Combobox(self.left,
                     textvariable=self.start_var,
                     values=self.states,
                     state="readonly",
                     width=20).grid(row=r, column=1)
        r += 1

        ttk.Label(self.left, text="End state:").grid(row=r, column=0, sticky="w")
        end_opts = ["(Same as start – round trip)"] + self.states
        self.end_var = tk.StringVar(value=end_opts[0])
        ttk.Combobox(self.left,
                     textvariable=self.end_var,
                     values=end_opts,
                     state="readonly",
                     width=20).grid(row=r, column=1)
        r += 1

        ttk.Label(self.left, text="Population:").grid(row=r, column=0, sticky="w")
        self.pop_var = tk.IntVar(value=100)
        ttk.Entry(self.left, textvariable=self.pop_var, width=10).grid(row=r, column=1, sticky="w")
        r += 1

        ttk.Label(self.left, text="Generations:").grid(row=r, column=0, sticky="w")
        self.gen_var = tk.IntVar(value=300)
        ttk.Entry(self.left, textvariable=self.gen_var, width=10).grid(row=r, column=1, sticky="w")
        r += 1

        ttk.Label(self.left, text="Mutation rate:").grid(row=r, column=0, sticky="w")
        self.mut_var = tk.DoubleVar(value=0.05)
        ttk.Entry(self.left, textvariable=self.mut_var, width=10).grid(row=r, column=1, sticky="w")
        r += 1

        ttk.Label(self.left, text="Crossover:").grid(row=r, column=0, sticky="w")
        self.cross_var = tk.StringVar(value="OX")
        ttk.Combobox(self.left,
                     textvariable=self.cross_var,
                     values=["OX", "PMX"],
                     state="readonly",
                     width=10).grid(row=r, column=1, sticky="w")
        r += 1

        ttk.Label(self.left, text="Cost per mile:").grid(row=r, column=0, sticky="w")
        self.cost_var = tk.DoubleVar(value=1.0)
        ttk.Entry(self.left, textvariable=self.cost_var, width=10).grid(row=r, column=1, sticky="w")
        r += 1

        ttk.Label(self.left, text="Travel speed (mph):").grid(row=r, column=0, sticky="w")
        self.speed_var = tk.DoubleVar(value=60.0)
        ttk.Entry(self.left, textvariable=self.speed_var, width=10).grid(row=r, column=1, sticky="w")
        r += 1

        ttk.Button(self.left,
                   text="Run Experiments",
                   command=self.run_all).grid(row=r, column=0, columnspan=2,
                                              sticky="ew", pady=(8, 10))
        r += 1

        ttk.Label(self.left, text="Plots:").grid(row=r, column=0, sticky="w")
        r += 1

        buttons = [
            ("NN Tour Map", self.show_nn),
            ("GA Tour Map", self.show_ga),
            ("GA Convergence", self.show_ga_conv),
            ("NN Convergence", self.show_nn_conv),
            ("NN vs GA Distance", self.show_dist_cmp),
            ("NN vs GA Compute Time", self.show_cmp_time),
            ("NN vs GA Travel Time", self.show_travel_cmp),
            ("NN vs GA Cost", self.show_cost_cmp),
            ("NN vs GA Fitness", self.show_fit_cmp),
            ("NN vs GA Boxplot", self.show_box),
        ]

        for text, cmd in buttons:
            ttk.Button(self.left, text=text, command=cmd).grid(
                row=r, column=0, columnspan=2, sticky="ew"
            )
            r += 1

        ttk.Button(self.left,
                   text="Summary Report",
                   command=self.show_summary).grid(row=r, column=0, columnspan=2,
                                                   sticky="ew", pady=(8, 0))
        r += 1

        ttk.Button(self.left,
                   text="Exit",
                   command=self.root.destroy).grid(row=r, column=0, columnspan=2,
                                                   sticky="ew", pady=(8, 0))

    def _reset_plot_area(self):
        self.msg = ttk.Label(self.plot_area, text="Run experiments first.", padding=10)
        self.msg.pack()

    def _draw_fig(self, fig):
        """Draw a matplotlib figure inside the Tk frame."""
        plt.close("all")
        if self.canvas is not None:
            self.canvas.get_tk_widget().destroy()
        self.canvas = FigureCanvasTkAgg(fig, master=self.plot_area)
        self.canvas.draw()
        self.canvas.get_tk_widget().pack(fill="both", expand=True)

    def _ready(self) -> bool:
        """Check that both NN and GA have been run before plotting."""
        if self.nn_route is None or self.ga_route is None:
            messagebox.showwarning("Warning", "Run experiments first.")
            return False
        return True

    # -------------------------------------------------
    # Run NN + GA
    # -------------------------------------------------
    def run_all(self):
        # decode start state
        try:
            self.start_idx = self.states.index(self.start_var.get())
        except ValueError:
            messagebox.showerror("Error", "Invalid start state.")
            return

        # decode end state
        end_choice = self.end_var.get()
        if end_choice.startswith("("):   # round trip
            self.end_idx = self.start_idx
            self.round_trip = True
        else:
            try:
                self.end_idx = self.states.index(end_choice)
            except ValueError:
                messagebox.showerror("Error", "Invalid end state.")
                return
            self.round_trip = (self.start_idx == self.end_idx)

        # numeric parameters
        try:
            pop = int(self.pop_var.get())
            gens = int(self.gen_var.get())
            mrate = float(self.mut_var.get())
            self.cost_per_mile = float(self.cost_var.get())
            self.speed = float(self.speed_var.get())
        except Exception:
            messagebox.showerror("Error", "Check GA, cost and speed values.")
            return

        if pop <= 0 or gens <= 0:
            messagebox.showerror("Error", "Population and generations must be > 0.")
            return

        # ------------- Nearest Neighbor -------------
        t0 = time.perf_counter()
        self.nn_route = nearest_neighbor(self.D, self.start_idx, self.end_idx, self.round_trip)
        self.nn_time = time.perf_counter() - t0
        self.nn_miles = path_length(self.nn_route, self.D, self.round_trip)
        self.nn_curve = nn_convergence(self.nn_route, self.D, self.round_trip)
        self.nn_fit = 1.0 / (1.0 + self.nn_miles)

        # ------------- Genetic Algorithm -------------
        runs = 10
        self.ga_dists = []
        self.ga_times = []
        best_len = float("inf")
        best_route = None
        best_hist = None

        for r in range(runs):
            seed = 1000 + r
            t0 = time.perf_counter()
            route, dist, hist = run_ga(
                self.D,
                self.start_idx,
                self.end_idx,
                self.round_trip,
                pop,
                gens,
                mrate,
                self.cross_var.get(),
                seed=seed
            )
            t1 = time.perf_counter()

            self.ga_dists.append(dist)
            self.ga_times.append(t1 - t0)

            if dist < best_len:
                best_len = dist
                best_route = route
                best_hist = hist

        self.ga_route = best_route
        self.ga_miles = best_len
        self.ga_hist = best_hist
        self.ga_fit = 1.0 / (1.0 + best_len)

        self.msg.config(text="Experiments done. Choose a plot on the left.")
        messagebox.showinfo("Done", "Finished running NN and GA.")

    # -------------------------------------------------
    # Route plotting helper
    # -------------------------------------------------
    def _plot_route(self, route: List[int], title: str):
        if not self._ready():
            return

        fig, ax = plt.subplots(figsize=(8, 6), dpi=110)

        if self.round_trip:
            draw_route = route + [route[0]]
        else:
            draw_route = route[:]

        xs = [self.coords[i][1] for i in draw_route]
        ys = [self.coords[i][0] for i in draw_route]

        ax.plot(xs, ys, "-o", markersize=4)

        # label each visited state once
        for idx in set(draw_route):
            ax.text(self.coords[idx][1], self.coords[idx][0], self.states[idx],
                    fontsize=6, ha="center")

        # start
        sx, sy = self.coords[self.start_idx][1], self.coords[self.start_idx][0]
        ax.scatter([sx], [sy], s=100, color="green", edgecolors="black")
        ax.text(sx, sy, "Start", fontsize=9)

        # end
        if self.round_trip:
            ex, ey = sx, sy
        else:
            ex, ey = self.coords[self.end_idx][1], self.coords[self.end_idx][0]
        ax.scatter([ex], [ey], s=100, color="red", edgecolors="black")
        ax.text(ex, ey, "End", fontsize=9)

        ax.set_title(title)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.grid(True, linestyle="--")
        ax.set_aspect("equal")

        plt.tight_layout()
        self._draw_fig(fig)

    # -------------------------------------------------
    # Plotting actions
    # -------------------------------------------------
    def show_nn(self):
        self._plot_route(self.nn_route, f"NN Route ({self.nn_miles:.2f} miles)")

    def show_ga(self):
        self._plot_route(self.ga_route, f"GA Best Route ({self.ga_miles:.2f} miles)")

    def show_ga_conv(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(self.ga_hist, color="green")
        ax.set_title("GA Convergence")
        ax.set_xlabel("Generation")
        ax.set_ylabel("Best Distance (miles)")
        ax.grid(True)
        plt.tight_layout()
        self._draw_fig(fig)

    def show_nn_conv(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(7, 4))
        steps = range(1, len(self.nn_curve) + 1)
        ax.plot(steps, self.nn_curve, marker="o", markersize=3)
        ax.set_title("NN Cumulative Distance")
        ax.set_xlabel("Step")
        ax.set_ylabel("Distance (miles)")
        ax.grid(True)
        plt.tight_layout()
        self._draw_fig(fig)

    def show_dist_cmp(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar(["NN", "GA"], [self.nn_miles, self.ga_miles])
        ax.set_title("Distance Comparison")
        ax.set_ylabel("Miles")
        plt.tight_layout()
        self._draw_fig(fig)

    def show_cmp_time(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar(["NN", "GA"], [self.nn_time, float(np.mean(self.ga_times))])
        ax.set_ylabel("Seconds")
        ax.set_title("Compute Time")
        plt.tight_layout()
        self._draw_fig(fig)

    def show_travel_cmp(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(5, 4))
        nn_h = self.nn_miles / self.speed
        ga_h = self.ga_miles / self.speed
        ax.bar(["NN", "GA"], [nn_h, ga_h])
        ax.set_ylabel("Hours")
        ax.set_title("Estimated Travel Time")
        plt.tight_layout()
        self._draw_fig(fig)

    def show_cost_cmp(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(5, 4))
        nn_c = self.nn_miles * self.cost_per_mile
        ga_c = self.ga_miles * self.cost_per_mile
        ax.bar(["NN", "GA"], [nn_c, ga_c])
        ax.set_title("Travel Cost Comparison")
        plt.tight_layout()
        self._draw_fig(fig)

    def show_fit_cmp(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar(["NN", "GA"], [self.nn_fit, self.ga_fit])
        ax.set_title("Fitness Comparison")
        plt.tight_layout()
        self._draw_fig(fig)

    def show_box(self):
        if not self._ready():
            return
        fig, ax = plt.subplots(figsize=(5, 4))
        nn_vals = [self.nn_miles] * len(self.ga_dists)
        ax.boxplot([nn_vals, self.ga_dists], labels=["NN", "GA"])
        ax.set_title("Distance Distribution (GA runs)")
        plt.tight_layout()
        self._draw_fig(fig)

    # -------------------------------------------------
    # Summary window
    # -------------------------------------------------
    def show_summary(self):
        if not self._ready():
            return

        win = tk.Toplevel(self.root)
        win.title("Summary Report")
        win.geometry("780x440")

        nn_hours = self.nn_miles / self.speed
        ga_hours = self.ga_miles / self.speed
        nn_cost = self.nn_miles * self.cost_per_mile
        ga_cost = self.ga_miles * self.cost_per_mile

        header_txt = (
            f"Start: {self.states[self.start_idx]}\n"
            f"End:   {self.states[self.start_idx] if self.round_trip else self.states[self.end_idx]}\n"
            f"Cost per mile: {self.cost_per_mile:.2f}\n"
            f"Speed: {self.speed:.2f} mph\n"
        )
        ttk.Label(win, text=header_txt, justify="left").pack(anchor="w", padx=10, pady=10)

        columns = ("Method", "Miles", "Compute(s)", "Travel(hrs)", "Cost", "Fitness")
        tree = ttk.Treeview(win, columns=columns, show="headings", height=4)
        for col in columns:
            tree.heading(col, text=col)
            tree.column(col, width=120, anchor="center")

        tree.insert(
            "",
            "end",
            values=(
                "NN",
                f"{self.nn_miles:.2f}",
                f"{self.nn_time:.4f}",
                f"{nn_hours:.2f}",
                f"{nn_cost:.2f}",
                f"{self.nn_fit:.6f}",
            ),
        )

        tree.insert(
            "",
            "end",
            values=(
                "GA",
                f"{self.ga_miles:.2f}",
                f"{float(np.mean(self.ga_times)):.4f}",
                f"{ga_hours:.2f}",
                f"{ga_cost:.2f}",
                f"{self.ga_fit:.6f}",
            ),
        )
        tree.pack(fill="x", padx=10)

        improvement = self.nn_miles - self.ga_miles
        pct = (improvement / self.nn_miles) * 100 if self.nn_miles else 0.0

        txt = tk.Text(win, height=6)
        txt.pack(fill="both", expand=True, padx=10, pady=10)
        txt.insert(
            "1.0",
            f"GA improvement over NN: {improvement:.2f} miles ({pct:.2f}%).\n"
            f"NN travel time: {nn_hours:.2f} hrs, GA travel time: {ga_hours:.2f} hrs.\n"
            f"NN cost: {nn_cost:.2f}, GA cost: {ga_cost:.2f}.\n"
            f"GA usually needs more compute time but finds a shorter overall route.\n",
        )
        txt.config(state="disabled")


# =========================================================
# 5. Main (safe in Jupyter / scripts)
# =========================================================

try:
    root.destroy()
except Exception:
    pass

root = tk.Tk()
root.geometry("1350x780")
app = TSPGui(root)
root.mainloop()